In [1]:
#Instalar los requerimientos de Huggin Face
!pip install -U transformers

## Transcripción
Usando el modelo Whisper de OpenAI (https://huggingface.co/openai/whisper-large-v3-turbo) creamos una pipeline de Hugging Face dedicada a la transcripción.

Las pipelines son objetos que simplifican el uso de distintos modelos de Hugging Face. Existen varios tipos, diferenciados según la tarea a la que están orientados.

Cada tarea requiere distintos parámetros. Por ejemplo, el parámetro return_timestamps obtiene el tiempo (en segundos) en que empieza y termina cada frase transcrita. Este es un parámetro específico de la pipeline de reconocimiento de voz.

In [ ]:
#Crear modelo
from transformers import pipeline #La pipeline ya te prepara todo para que se pueda usar , es una manera de ejcutarlo , solo hay que especificar lo de abajo y ya

pipe = pipeline("automatic-speech-recognition", model="openai/whisper-large-v3-turbo")

In [ ]:
import os
import librosa # libreria para manejar audio
import json

def transcribe_audio_to_json(filename):
    cwd = os.getcwd()
    audio_path = os.path.join(cwd, filename)

    try:
        # Leer el audio usando librosa
        audio_data, sample_rate = librosa.load(audio_path, sr=16000)  
        # En principio funciona tambien con formato video

        # Usamos la pipeline con el argumento de return_timestamps=True
        # así obtenemos el diálogo segmentado con cuándo empieza y acaba
        result = pipe(audio_data, return_timestamps=True)
        # Eliminamos la extensión del nombre del archivo (.mp3, .wav...)
        json_name = f"{os.path.splitext(filename)[0]}.json"
        json_dir = os.path.join(cwd, "json")
        if not os.path.isdir(json_dir):
            os.makedirs(json_dir)
        json_path = os.path.join(json_dir, json_name)

        try:
            #  Guardar en un archivo json
            with open(json_path, "w", encoding="utf-8") as archivo:
                archivo.write(json.dumps(result))
            print(f"\033[32mArchivo guardado: {json_path}\033[0m")   
        except FileNotFoundError:
            # Si la carpeta "json" no existe
            print(f"Error: El directorio no existe {json_path}")
        except Exception as e:
            print(f"Error procesando audio: {e}")

    # Manejo de errores
    except FileNotFoundError:
        print(f"Error: No se encontró el archivo {audio_path}")
    except Exception as e:
        print(f"Error procesando audio: {e}")



In [6]:
import json

def read_json_chunks(json_path):
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            transcript = json.loads(f.read())
            return transcript['chunks']
    except Exception as e:
        print(f"Error: {e}")


## Crear los subtítulos
El formato estándar para subtítulos es srt, un archivo de texto plano con esta estructura:

```
1
00:02:16,612 --> 00:02:19,376
Senator, we're making
our final approach into Coruscant.

2
00:02:19,482 --> 00:02:21,609
Very good, Lieutenant.

3
00:03:13,336 --> 00:03:15,167
We made it.
```

Tenemos que adaptar los fragmentos de texto con sus tiempos en segundos a esta estructura como describiremos en el código a continuación.

In [19]:
# Interpretamos el diccionario con el resultado:
# descartamos la transcripción entera (text) y nos guardamos los chunks
chunks = res["chunks"]

# Chunks (fragmentos) es un array
# Cada chunk tiene:
# - timestamp, una tupla con el tiempo donde empieza y acaba la frase
# - text, la transcripción de la frase

# Hay que adaptar las timestamps al formato estándar:
# 00:00:00,000 horas:minutos:segundos,milisegundos
def format_timestamp(timestamp):
    hours = int(timestamp // 3600)
    minutes = int((timestamp % 3600) // 60)
    seconds = int(timestamp % 60)
    milliseconds = int(round((timestamp - int(timestamp)) * 1000))

    # :02d es un especificador de formato para un string:
    # 0 -> cubre el tamaño con 0
    # 2 -> el string siempre tamaño 2
    # d -> es un número decimal, un int
    return f"{hours:02d}:{minutes:02d}:{seconds:02d},{milliseconds:03d}"

# El formato de un srt línea a línea por cada fragmento es:
# El número del fragmento empezando por 1 (un contador)
# La marca de tiempo donde empieza y acaba 00:00:00,000 --> 00:00:00,000
# El texto (puede estar separado en varias líneas)
# Un salto de línea para separar entre fragmentos
def format_srt(chunks):
    srt = ""
    num = 1
    for chunk in chunks:
        start = format_timestamp(chunk["timestamp"][0])
        end = format_timestamp(chunk["timestamp"][1])
        text = chunk["text"]
        srt += f"{num}\n{start} --> {end}\n{text}\n\n"
        num += 1
    return srt


In [ ]:
# Guardamos el string que acabamos de formatear como srt
import os

def save_srt(srt, filename):
    dir = os.path.join(os.getcwd(), "subs")
    path = os.path.join(dir, filename + ".srt")
    try:
        if not os.path.isdir(dir):
            os.makedirs(dir)
        
        with open(path, "w") as f:
            f.write(srt)
    except FileNotFoundError:
        print(f"Error: El directorio no existe {path}")
    except Exception as e:
        print(f"Error guardando srt: {e}")

save_srt(format_srt(chunks), "audio_01")